# Mount Azure Data Lake Storage (ADLS Gen2) in Databricks

This notebook shows how to **mount an Azure Data Lake Storage Gen2 container** so you can read and write data using DBFS paths in later labs.

---

Set your **student_id** in Step 1 below; the notebook will mount to `/mnt/data-<student_id>`. Attach this notebook to your cluster.

---

## Prerequisites

1. **Azure Data Lake Storage Gen2** (or a storage account with hierarchical namespace enabled).
2. A **container** (e.g. `inputdata`) with your data uploaded.
3. **Service Principal (SP)** in Azure AD with at least **Storage Blob Data Contributor** (or Reader) on the container/storage account.
4. **Databricks:** Secret scope with SP credentials (recommended), or use variables below for lab only (do not commit secrets).

## Step 1 — Your student ID and Data Lake account

In [ ]:
# Your student ID (u01–u16). Mount point will be /mnt/data-<student_id>
student_id = "u01"  # e.g. "u05" for student u05

# Replace with your Azure Storage account name (ADLS Gen2)
adlsAccountName = "your-adls-account-name"
adlsContainerName = "your-container-name"
adlsFolderName = "data"  # optional subfolder under container

## Step 2 — Service Principal credentials

In [ ]:
# Service Principal: Application (client) ID, Client secret, Tenant ID
# Replace with your values or use dbutils.secrets.get() as in Option A
applicationId = "your-application-id"
authenticationKey = "your-client-secret"
tenandId = "your-tenant-id"

## Step 3 — Build OAuth endpoint and ABFSS source

In [ ]:
endpoint = "https://login.microsoftonline.com/" + tenandId + "/oauth2/token"
source = "abfss://" + adlsContainerName + "@" + adlsAccountName + ".dfs.core.windows.net/"
print("Source (ABFSS):", source)

## Step 4 — Mount configuration

In [ ]:
configs = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": applicationId,
    "fs.azure.account.oauth2.client.secret": authenticationKey,
    "fs.azure.account.oauth2.client.endpoint": endpoint
}

## Step 5 — Unmount (if already mounted) and mount

Mount point is **`/mnt/data-<student_id>`** (e.g. `/mnt/data-u05`). After this, paths like `dbfs:/mnt/data-u05/your-folder/` map to your container.

In [ ]:
# Separate mount point per student (e.g. /mnt/data-u05)
mount_point = "/mnt/data-" + student_id

try:
    dbutils.fs.unmount(mount_point)
    print(mount_point, "unmounted (was already mounted).")
except Exception as e:
    print("Unmount skipped:", str(e))

dbutils.fs.mount(source=source, mount_point=mount_point, extra_configs=configs)
print("Mounted:", source, "->", mount_point)

## Step 6 — Verify mount (lists your mount point)

In [ ]:
display(dbutils.fs.ls(mount_point))